# Dataset setup and provenance

Documents the frozen corpus sources, revisions, passage count, and checksums. Downloads are intentionally not triggered from a notebook.

**Status:** provisional development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
manifest = json.loads((ROOT / "artifacts/metadata/phase1_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest, ensure_ascii=False, indent=2))


{
  "articles": {
    "articles": 4000,
    "domains": {
      "culture": 460,
      "general": 900,
      "geography": 938,
      "history": 757,
      "pakistan": 748,
      "science": 197
    },
    "path": "data/raw/wikipedia.jsonl",
    "sha256": "cf8abe3bc85140e7d8fe202d0406085214942d736bd16f3eeb92a5b95022906c",
    "tokens": {
      "maximum": 45533,
      "mean": 463.0,
      "median": 186.0,
      "minimum": 80
    }
  },
  "passage_variants": [
    {
      "passages": 20106,
      "path": "data/processed/passages_120_24.jsonl",
      "represented_articles": 4000,
      "sha256": "67ddd9fdeef1f7bbf48026ca8cffcbf52a13aeba163b3fb8250b5a2eb03d1463",
      "tokens": {
        "maximum": 120,
        "mean": 111.34,
        "median": 120.0,
        "minimum": 25
      }
    },
    {
      "passages": 16352,
      "path": "data/processed/passages_150_30.jsonl",
      "represented_articles": 4000,
      "sha256": "47648cf679facb9a576841542289767f854c1a27aca6cb8e14e3f3eb1a2e5671",
   

## Artifact integrity


In [3]:
manifest = json.loads((ROOT / 'artifacts/metadata/phase1_manifest.json').read_text(encoding='utf-8'))
checks = []
for item in manifest['passage_variants']:
    path = ROOT / item['path']
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checks.append({'file': path.name, 'expected_passages': item['passages'], 'actual_passages': sum(1 for line in path.open(encoding='utf-8') if line.strip()), 'checksum_ok': digest == item['sha256']})
print(json.dumps(checks, indent=2))
assert all(row['expected_passages'] == row['actual_passages'] and row['checksum_ok'] for row in checks)


[
  {
    "file": "passages_120_24.jsonl",
    "expected_passages": 20106,
    "actual_passages": 20106,
    "checksum_ok": true
  },
  {
    "file": "passages_150_30.jsonl",
    "expected_passages": 16352,
    "actual_passages": 16352,
    "checksum_ok": true
  },
  {
    "file": "passages_180_36.jsonl",
    "expected_passages": 13908,
    "actual_passages": 13908,
    "checksum_ok": true
  }
]


## Diagnostic split audit


In [4]:
with (ROOT / 'data/diagnostic/raabta_diagnostic_codex.csv').open(encoding='utf-8-sig', newline='') as handle:
    rows = list(csv.DictReader(handle))
print('Questions:', len(rows))
print('Split:', dict(Counter(row['split'] for row in rows)))
print('Query types:', dict(sorted(Counter(row['query_type'] for row in rows).items())))
assert Counter(row['split'] for row in rows) == {'development': 120, 'test': 60}


Questions: 180
Split: {'development': 120, 'test': 60}
Query types: {'abbreviated_roman_urdu': 23, 'clean_roman_urdu': 23, 'highly_noisy_roman_urdu': 23, 'informal_spelling': 23, 'named_entity': 22, 'short_query': 22, 'slightly_ambiguous': 22, 'urdu_english_code_switching': 22}


## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
